# 1.0 Importando pacotes

In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
import regex as re

# 2.0 Definindo pacotes

In [2]:
# Caminho da pasta de inputs
inputs_path = Path('./inputs').resolve()

# Caminho da pasta de outputs
outputs_path = Path('./outputs').resolve()

# Caminhos dos subsets
co_path = inputs_path / 'subset_co.parquet'
no2_path = inputs_path / 'subset_no2.parquet'
o3_path = inputs_path / 'subset_o3.parquet'
pm_path = inputs_path / 'subset_pm.parquet'
so2_path = inputs_path / 'subset_so2.parquet'

# 3.0 Carregando subsets

In [3]:
subset_co = gpd.read_parquet(co_path)
subset_no2 = gpd.read_parquet(no2_path)
subset_o3 = gpd.read_parquet(o3_path)
subset_pm = gpd.read_parquet(pm_path)
subset_so2 = gpd.read_parquet(so2_path)

## 4.0  Classificando a representatividade espacial de todas as estações

In [4]:
# %% CLASSIFICANDO O STATUS DE REPRESENTATIVIDADE DE CADA ESTAÇÃO ----------------------------
def classify_spatial_rep(subset):
    """
    Verifica valores True nas colunas referentes a cada escala espacial
    e atribui a cada estação a escala mais restritiva, seguindo a ordem abaixo:    
                      1º  >   2º   >    3º    >   4º
                    MICRO > MESO > BAIRRO > URBANA
    
    A lógica é que, se houver pelo menos uma via que determine que a estação é 
    representativa espacialmente para uma classe mais restritiva, então ela
    é considerada representativa nessa escala.

    Parâmetros
    ----------
    subset : geodataframe 
        Subconjunto das estações de monitoramento, referente a um único poluente

    Retorna
    -------
    subset : geodataframe
        Subconjunto de entrada com uma nova coluna chamada 'REP_ESPACIAL_NAME'
    """

    # Assegurando de não estar sobrescrevendo o subset original
    subset = subset.copy()
    
    # Criando lista de possíveis colunas a considerar
    # A lista foi criada do menos restritivo para mais restritivo
    rep_possibilities = {
        'rep_urb_any': 'urbana',
        'rep_bairro_any': 'bairro',
        'rep_meso_any': 'meso',
        'rep_micro_any': 'micro',
    }

    # Filtrando as que estão no subset
    rep_possibilities = {
        col: item
        for col, item in rep_possibilities.items()
        if col in subset.columns
    }

    # Definindo variável de resultado como lista vazia do tamanho de subset
    rep_spatial_name_serie = pd.Series(index=range(len(subset)), dtype='str')
    rep_spatial_name_serie.loc[:] = 'urbana'

    # Adicionando resultados do menos restritivo pro mais restritivo
    for header, item in rep_possibilities.items():
        # Adicionando só onde os valores da coluna forem iguais a True
        rep_spatial_name_serie.loc[subset[header]] = item

    # Adicionando no subset final
    subset.loc[:, 'REP_ESPACIAL_NAME'] = rep_spatial_name_serie

    return subset


# 5.0 Aplicando função

In [5]:
subset_co = classify_spatial_rep(subset_co)
subset_no2 = classify_spatial_rep(subset_no2)
subset_o3 = classify_spatial_rep(subset_o3)
subset_pm = classify_spatial_rep(subset_pm)
subset_so2 = classify_spatial_rep(subset_so2)

In [6]:
# pd.concat([subset_co,subset_no2,subset_o3,subset_pm,subset_so2])['REP_ESPACIAL_NAME'].unique()

In [7]:
# subset_pm.loc[subset_pm.ID_OEMA.str.contains('Fercal Boa Vista')].iloc[[1]][[col for col in list(subset_pm.columns) if 'REP_ESPACIAL_NAME' in col]]

# 6.0 Identificando ID da via mais próxima que classifica a estação

In [8]:
def filtrar_via_mais_prox_valida(subset:gpd.GeoDataFrame):
    """
    Essa função identifica, dentre as vias que classificam cada estação, o osm_id
    da via mais próxima.

    Parameters
    ----------
    subset: gpd.GeoDataFrame
        Geodataframe de estações de monitoramento da qualidade do ar para um poluente
        específico, com as colunas 'rep_{classe}_{faixa}', 'distance_{faixa}' e 
        'REP_ESPACIAL_NAME', onde {classe} são as classes de representatividade e {faixa}
        são as faixas de adt existentes para o poluente.

    Returns
    --------
    subset: gpd.GeoDataFrame
        Geodataframe original com 
    
    """
    # Lista de classes de representatividade, na ordem de prioridade
    classes = ['micro','meso','bairro','urbana']

    # Classes existentes para as estações do poluente
    classes_validas = list(subset.loc[:,'REP_ESPACIAL_NAME'].unique())
    classes_validas = [c for c in classes if c in classes_validas]
    classes_validas = ['urb' if c == 'urbana' else c for c in classes_validas]

    for classe in classes_validas:
        ##1 e 2) ============= Filtrando faixas de ADT existentes na classe ================
        faixas_adt = []
        colunas = []
        distancias = []
        for col in subset.columns:
            match1 = re.search(f"rep_{classe}_(\d+k?)$", col)
            if match1:
                colunas.append(match1.group(0))
                faixas_adt.append(match1.group(1))
            match2 = re.search(f"distance_(\d+k?)$", col)
            if match2:
                distancias.append(match2.group(0))
        # print(f'\n1)Faixas de adt existentes para a classe <{classe}> -------------------------')
        # print(faixas)
        # print(f'\n2)Colunas de distancias para a classe <{classe}> -------------------------')
        # print(distancias)
    
        
        # 3) ================= Distancias ===================================
        # Adiciona uma coluna com as colunas de distância
        subset[f'distancias_{classe}'] = subset[distancias].values.tolist()
        
        # print(f'\n3)Distancias <{classe}> -------------------------------------')
        # print(subset[f'distancias_{classe}'].head(8))
    
        
        ## 4) ==============  Máscara de colunas de adt válidas =====================
        subset[f'cols_{classe}'] = subset[colunas].values.tolist()
    
        # print(f'\n4) Máscara de colunas válidas <{classe}> -------------------------------------')
        # print(subset[f'cols_{classe}'].head(8))
    
        
        ## 5) =================== Determinando faixas de adt  ======================
        subset[f'cols_validas_{classe}'] = [
            [faixa for (faixa, val) in zip(faixas_adt,linha) if val]
            for linha in subset[f'cols_{classe}']
        ]
        # print(f'\n5) Faixas de adt que validam a classe <{classe}> para a estação ---------------')
        # print(subset[f'cols_validas_{classe}'].head(8))
    
        
        # 6) ==================== Listando distâncias das vias mais próximas ================
        # Listando colunas de distancias validas
        subset[f'cols_distancias_validas_{classe}'] = [
            [distancia for (distancia,val) in zip(distancias, linha) if val]
            for linha in subset[f'cols_{classe}']
        ]
        # print(f'\n6) Colunas de distancias validas para a classe <{classe}> ---------------')
        # print(subset[f'cols_distancias_validas_{classe}'].head(8))
    
        # 7) =================== Pegando valores das distancias válidas =========================
        subset[f'distancias_validas_{classe}'] = [
            [subset.loc[idx,distancia] for (distancia,val) in zip(distancias,linha) if val]
            for idx,linha in enumerate(subset[f'cols_{classe}'])
        ]
        # print(f'\n7) Valores de distancias validas para a classe <{classe}> ---------------')
        # print(subset[f'distancias_validas_{classe}'].head(8))
    
        # 8) ============== Pegando a distancia da via mais proxima dentre as faixas ===================
        def pegar_min(linha):
            """
            Pega uma lista numerica e retorna o menor valor
            """
            if not linha:
                return np.nan
            else:
                return min(linha)
                
        subset[f'min_distancias_{classe}'] = subset[f'distancias_validas_{classe}'].apply(pegar_min)
    
        # print(f'\n8) Distancias minimas <{classe}> ---------------')
        # print(subset[f'min_distancias_{classe}'].head(8))
    
        # 9) =========== Pegando o indice da via mais proxima dentre as faixas na lista ===================
        def pegar_idx_do_min(linha):
            """
            Pega um lista numerica e retorna o indice do menor valor
            """
            if not linha:
                return np.nan
            else:
                linha_min_dist = min(linha)
                ind = [int(idx) for (idx, c) in enumerate(linha) 
                       if c == linha_min_dist][0]
                return int(ind)
            
        subset[f'min_idx_{classe}'] = (
            subset[f'distancias_validas_{classe}'].apply(pegar_idx_do_min).astype("Int64")
        )
    
        # print(f'\n9) Indice da lista da distancia minima <{classe}> ---------------')
        # print(subset[f'min_idx_{classe}'].head(8))
    
        # 10) ========== Pegando a faixa de adt da via mais proxima, pelo indice =========================
        for linha in subset.index:
            # faixa de adt no indice min_idx
            lista_faixas = subset.loc[linha,f'cols_distancias_validas_{classe}']
            if not lista_faixas:
                subset.loc[linha,f'faixa_dist_min_{classe}'] = np.nan
            else:
                subset.loc[linha,f'faixa_dist_min_{classe}'] = (
                    lista_faixas[subset.loc[linha,f'min_idx_{classe}']]
                )
    
        # print(f'\n10) Faixa de adt das distancias minimas <{classe}> ---------------')
        # print(subset[f'faixa_dist_min_{classe}'].head(8))
    
        # 11) ============ OSM_ID DA VIA MAIS PROXIMA VALIDA PARA A CLASSE ================================
        subset[f"osm_id_mais_prox_{classe}"] = subset.to_numpy()[
            np.arange(len(subset)),
            subset.columns.get_indexer(subset[f"faixa_dist_min_{classe}"].str.replace("distance_", "osm_id_"))
    ]
        # print(f'\n11) OSM_ID da via mais proxima válida <{classe}> ---------------')
        # print(subset[f'osm_id_mais_prox_{classe}'].head(8))
        # print(f'\n =============== FIM DE {classe.upper()} ==============================================')
    
    #12) ============ Pegando o valor da primeira coluna não Nan, na ordem de prioridade =====================
    subset["osm_id_mais_prox_valida"] = (
        subset[[f'osm_id_mais_prox_{classe}' for classe in classes_validas]]
        .bfill(axis=1)
        .iloc[:, 0]
    )
    # print(f'\n12) OSM_ID da via mais proxima válida total ---------------')
    # print(subset['osm_id_mais_prox_valida'].head(8))

    return subset
    

In [9]:
# Aplicando função
subset_co = filtrar_via_mais_prox_valida(subset_co)
subset_no2 = filtrar_via_mais_prox_valida(subset_no2)
subset_o3 = filtrar_via_mais_prox_valida(subset_o3)
subset_pm = filtrar_via_mais_prox_valida(subset_pm)
subset_so2 = filtrar_via_mais_prox_valida(subset_so2)

/tmp/ipykernel_700800/3352708949.py:130: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'distance_1k' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  subset.loc[linha,f'faixa_dist_min_{classe}'] = (
/tmp/ipykernel_700800/3352708949.py:149: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .bfill(axis=1)
/tmp/ipykernel_700800/3352708949.py:130: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'distance_15k' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  subset.loc[linha,f'faixa_dist_min_{classe}'] = (
/tmp/ipykernel_700800/335270894

## Capitalizando primeira letra da classe de representatividade espacial

In [10]:
subset_co.REP_ESPACIAL_NAME = subset_co.REP_ESPACIAL_NAME.apply(str.capitalize)
subset_no2.REP_ESPACIAL_NAME = subset_no2.REP_ESPACIAL_NAME.apply(str.capitalize)
subset_o3.REP_ESPACIAL_NAME = subset_o3.REP_ESPACIAL_NAME.apply(str.capitalize)
subset_so2.REP_ESPACIAL_NAME = subset_so2.REP_ESPACIAL_NAME.apply(str.capitalize)
subset_pm.REP_ESPACIAL_NAME = subset_pm.REP_ESPACIAL_NAME.apply(str.capitalize)

# 7.0 Salvando outputs

In [11]:
subset_co.to_parquet(outputs_path / 'subset_co.parquet')
subset_no2.to_parquet(outputs_path / 'subset_no2.parquet')
subset_o3.to_parquet(outputs_path / 'subset_o3.parquet')
subset_pm.to_parquet(outputs_path / 'subset_pm.parquet')
subset_so2.to_parquet(outputs_path / 'subset_so2.parquet')